# GEAP MCP Tool Servers — Create & Monitor (SDK-First Deep-Dive)

A companion to `platform_sdk_demo.ipynb`. **MCP tool servers** are how an agent acts on the world. This notebook goes deep on **authoring** a FastMCP server (and calling its tools live, in-process) and on **monitoring** it once deployed — Cloud Run request metrics, logs, and the tool-call spans that show up in the agent's traces.

**Legend.** ✅ runs live here (in-process tool calls, read-only metric queries). 🔒 shown-but-guarded: Cloud Run deploy + registry registration print the exact command and run only when `GEAP_RUN_DEPLOY=1`. 🔧 marks custom/infra.

> Create → deploy → register → monitor. Headless twins: `bash scripts/deploy_all.sh` (deploy + register) and the monitoring reads below (`monitoring_v3` / `gcloud logging`).

## Setup

In [1]:
import os, json, shlex, subprocess, time
for _ in range(6):
    if os.path.exists("src/config.py"):
        break
    os.chdir("..")

from src.config import GCP_PROJECT_ID, GCP_REGION
os.environ["CLOUDSDK_CORE_DISABLE_PROMPTS"] = "1"

RUN_DEPLOY = os.environ.get("GEAP_RUN_DEPLOY") == "1"       # gate Cloud Run deploy + registry register
MCP_SERVICE = "search-mcp"                                   # the service we author/deploy/monitor

def run_or_show(cmd, live: bool, timeout: int = 180):
    """🔧 Print a command; execute it (never raising) only when `live` is True."""
    printable = cmd if isinstance(cmd, str) else " ".join(shlex.quote(c) for c in cmd)
    print(f"$ {printable}")
    if not live:
        print("  🔒 skipped — set GEAP_RUN_DEPLOY=1 to run this for real")
        return None
    try:
        out = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout)
        print((out.stdout or out.stderr)[-2000:] or "  (no output)")
        return out
    except Exception as e:
        print(f"  (skipped/failed: {type(e).__name__}: {e})")
        return None

print("project:", GCP_PROJECT_ID, "| region:", GCP_REGION, "| service:", MCP_SERVICE)
print("guard -> RUN_DEPLOY:", RUN_DEPLOY)

project: wortz-project-352116 | region: us-central1 | service: search-mcp
guard -> RUN_DEPLOY: False


## Phase 1 — The FastMCP pattern (call tools in-process)
📖 [`src/mcp_servers/search/server.py`](https://github.com/jswortz/geap-tour/blob/main/src/mcp_servers/search/server.py)

An MCP server is a `FastMCP` instance with `@mcp.tool()`-decorated plain Python functions over a mock data layer, run on `streamable-http`. Because the decorator leaves the function callable, you can exercise the tools directly — no server, no network.

In [2]:
from src.mcp_servers.search import server as search_mcp

print("server:", search_mcp.mcp.name)
print("search_flights('SFO','JFK') ->", search_mcp.search_flights("SFO", "JFK")[0])
print("search_hotels('Miami')     ->", search_mcp.search_hotels("Miami")[0])
# Packaging: a per-server Dockerfile runs `python -m server` on streamable-http (ports 8001/8002/8003).

server: search-mcp
search_flights('SFO','JFK') -> {'id': 'FL001', 'airline': 'United', 'origin': 'SFO', 'destination': 'JFK', 'date': '2026-06-15', 'price': 450.0, 'departure': '08:00', 'arrival': '16:30'}
search_hotels('Miami')     -> {'id': 'HT003', 'name': 'Fontainebleau Miami', 'city': 'Miami', 'price_per_night': 400.0, 'rating': 4.7, 'available_from': '2026-06-01', 'available_to': '2026-12-31'}


## Phase 2 — Author a new tool
📖 [FastMCP](https://github.com/jlowin/fastmcp)

Authoring is one decorator on a typed function. Here's a brand-new server + tool, exercised in-process.

In [3]:
from fastmcp import FastMCP

weather_mcp = FastMCP("weather-mcp", instructions="Look up simple travel weather.")

@weather_mcp.tool()
def get_weather(city: str) -> dict:
    """Return a (mock) weather summary for a city."""
    table = {"Miami": {"hi": 88, "lo": 75, "cond": "humid"}, "NYC": {"hi": 61, "lo": 48, "cond": "clear"}}
    return {"city": city, **table.get(city, {"hi": 70, "lo": 55, "cond": "mild"})}

print("new tool call:", get_weather("Miami"))
print("To ship it: add a Dockerfile (EXPOSE 8004; CMD python -m server) and deploy like the others.")

new tool call: {'city': 'Miami', 'hi': 88, 'lo': 75, 'cond': 'humid'}
To ship it: add a Dockerfile (EXPOSE 8004; CMD python -m server) and deploy like the others.


## Phase 3 — Deploy to Cloud Run — 🔒 guarded
📖 [`src/deploy/deploy_mcp_servers.py`](https://github.com/jswortz/geap-tour/blob/main/src/deploy/deploy_mcp_servers.py)

Each server deploys to Cloud Run from its source dir; the workshop smoke-tests it with a JSON-RPC `initialize` POST to `/mcp`.

In [4]:
run_or_show(["gcloud", "run", "deploy", MCP_SERVICE,
             "--source", f"src/mcp_servers/{MCP_SERVICE.replace('-mcp','')}",
             "--region", GCP_REGION, "--project", GCP_PROJECT_ID,
             "--port", "8001", "--allow-unauthenticated", "--min-instances", "1", "--quiet"], RUN_DEPLOY)
# ...or deploy all three at once:  bash scripts/deploy_all.sh

$ gcloud run deploy search-mcp --source src/mcp_servers/search --region us-central1 --project wortz-project-352116 --port 8001 --allow-unauthenticated --min-instances 1 --quiet
  🔒 skipped — set GEAP_RUN_DEPLOY=1 to run this for real


## Phase 4 — Register to the Agent Registry — 🔒 guarded
📖 see [`registry_sdk_demo.ipynb`](registry_sdk_demo.ipynb)

Registering the deployed server (from its toolspec) lets agents discover it by name instead of a hardcoded URL. Full detail in the registry notebook.

In [5]:
run_or_show(["gcloud", "alpha", "agent-registry", "services", "create", MCP_SERVICE,
             "--project", GCP_PROJECT_ID, "--location", GCP_REGION, "--display-name", MCP_SERVICE,
             "--mcp-server-spec-type=tool-spec",
             "--mcp-server-spec-content=scripts/toolspecs/search_toolspec.json",
             "--interfaces=url=https://search-mcp-xxxx-uc.a.run.app/mcp,protocolBinding=JSONRPC"], RUN_DEPLOY)

$ gcloud alpha agent-registry services create search-mcp --project wortz-project-352116 --location us-central1 --display-name search-mcp --mcp-server-spec-type=tool-spec --mcp-server-spec-content=scripts/toolspecs/search_toolspec.json --interfaces=url=https://search-mcp-xxxx-uc.a.run.app/mcp,protocolBinding=JSONRPC
  🔒 skipped — set GEAP_RUN_DEPLOY=1 to run this for real


## Phase 5 — Monitor the server
📖 [`docs/monitoring_integration_guide.md`](https://github.com/jswortz/geap-tour/blob/main/docs/monitoring_integration_guide.md)

Three lenses: **Cloud Run request metrics** (throughput/latency, via `monitoring_v3`), **logs** (`cloud_run_revision`), and **tool-call spans** inside the agent's OTel traces (Cloud Trace). We read request metrics live (read-only).

In [6]:
from google.cloud import monitoring_v3

def read_run_metric(service: str, metric: str = "request_count", hours: int = 24):
    """🔧 Read a Cloud Run metric for one service from Cloud Monitoring (read-only)."""
    mc = monitoring_v3.MetricServiceClient()
    now = int(time.time())
    interval = monitoring_v3.TimeInterval(start_time={"seconds": now - hours * 3600}, end_time={"seconds": now + 5})
    it = mc.list_time_series(request={
        "name": f"projects/{GCP_PROJECT_ID}",
        "filter": f'metric.type="run.googleapis.com/{metric}" AND resource.labels.service_name="{service}"',
        "interval": interval,
        "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
    })
    return [(p.value.int64_value or p.value.double_value) for ts in it for p in ts.points]

try:
    pts = read_run_metric(MCP_SERVICE, "request_count")
    print(f"{MCP_SERVICE}: {len(pts)} request_count points in last 24h; total requests ~ {sum(pts):.0f}")
except Exception as e:
    print(f"(request metrics need the deployed service + monitoring access — {type(e).__name__}: {e})")

# Logs (guarded — needs the deployed service):
run_or_show(["gcloud", "logging", "read",
             f'resource.type="cloud_run_revision" AND resource.labels.service_name="{MCP_SERVICE}"',
             "--project", GCP_PROJECT_ID, "--limit", "5"], RUN_DEPLOY)

print("\nTool-call traces: OTEL_ENV_VARS on the agent send tool spans to Cloud Trace; drive traffic with")
print("  uv run python -m src.traffic.generate_traffic   # exercises search_flights / search_hotels / submit_expense")

search-mcp: 648 request_count points in last 24h; total requests ~ 1377
$ gcloud logging read 'resource.type="cloud_run_revision" AND resource.labels.service_name="search-mcp"' --project wortz-project-352116 --limit 5
  🔒 skipped — set GEAP_RUN_DEPLOY=1 to run this for real

Tool-call traces: OTEL_ENV_VARS on the agent send tool spans to Cloud Trace; drive traffic with
  uv run python -m src.traffic.generate_traffic   # exercises search_flights / search_hotels / submit_expense


## Phase 6 — Alert on MCP health — 🔒 guarded
📖 [`src/eval/quality_alerts.py`](https://github.com/jswortz/geap-tour/blob/main/src/eval/quality_alerts.py)

Turn a Cloud Run metric into a Cloud Monitoring alert (e.g. p99 request latency). We build the `AlertPolicy` object here (pure SDK, no cloud call) and create it only when guarded.

In [7]:
from google.cloud import monitoring_v3
from google.protobuf import duration_pb2

condition = monitoring_v3.AlertPolicy.Condition(
    display_name=f"{MCP_SERVICE} p99 request latency > 2s",
    condition_threshold=monitoring_v3.AlertPolicy.Condition.MetricThreshold(
        filter=(f'metric.type="run.googleapis.com/request_latencies" '
                f'AND resource.type="cloud_run_revision" AND resource.labels.service_name="{MCP_SERVICE}"'),
        comparison=monitoring_v3.ComparisonType.COMPARISON_GT,
        threshold_value=2000,   # milliseconds
        duration=duration_pb2.Duration(seconds=300),
        aggregations=[monitoring_v3.Aggregation(
            alignment_period=duration_pb2.Duration(seconds=300),
            per_series_aligner=monitoring_v3.Aggregation.Aligner.ALIGN_PERCENTILE_99)],
    ),
)
policy = monitoring_v3.AlertPolicy(
    display_name=f"GEAP MCP Health - {MCP_SERVICE} latency",
    conditions=[condition],
    combiner=monitoring_v3.AlertPolicy.ConditionCombinerType.OR, enabled=True)
print("built AlertPolicy:", policy.display_name, "| condition:", condition.display_name)

if RUN_DEPLOY:
    client = monitoring_v3.AlertPolicyServiceClient()
    created = client.create_alert_policy(name=f"projects/{GCP_PROJECT_ID}", alert_policy=policy)
    print("created:", created.name)
else:
    print("🔒 skipped create_alert_policy (set GEAP_RUN_DEPLOY=1 to create it)")

built AlertPolicy: GEAP MCP Health - search-mcp latency | condition: search-mcp p99 request latency > 2s
🔒 skipped create_alert_policy (set GEAP_RUN_DEPLOY=1 to create it)


## Recap

- **Create** (✅): a FastMCP server = `@mcp.tool()` on typed functions + a mock data layer; tools are callable in-process for fast iteration.
- **Deploy** (🔒): `gcloud run deploy` per server (or `scripts/deploy_all.sh`).
- **Register** (🔒): `gcloud alpha agent-registry services create` from a toolspec — see `registry_sdk_demo.ipynb`.
- **Monitor** (✅/🔒): Cloud Run request metrics via `monitoring_v3`, `cloud_run_revision` logs, tool-call spans in Cloud Trace, and a latency `AlertPolicy`.

Govern which agents may call this server in **`gateway_sdk_demo.ipynb`** (policies) and **`registry_sdk_demo.ipynb`** (bindings).